In [1]:
import numpy as np
import pandas as pd
from aiwhatif_cf.config import DATA_DIR, CF_OUTPUTS

batch_cfcheck_data = CF_OUTPUTS / "xgboost_2026-04-14" / "random_annotated_hltprhc.csv"
batch_cfcheck_df = pd.read_csv(batch_cfcheck_data)

In [2]:
pd.set_option("display.max_rows", None)

In [3]:
batch_cfcheck_df = batch_cfcheck_df.drop(columns=["hltprhc", "target_risk"])

In [4]:

batch_cfcheck_df["cf_id"] = batch_cfcheck_df["cf_id"].replace({"original": "xin"})

batch_cfcheck_df = batch_cfcheck_df.rename(columns={
    "original_risk": "risk_before",
    "predicted_risk" : "predicted_risk_after",
    "meets_target_risk" : "valid",
})

batch_cfcheck_df.head(4)

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,risk_before,predicted_risk_after,valid
0,0,xin,2.0,3.0,6.0,3.0,1.0,0.0,20.438166,2.0,0.056522,NaN,NaN
1,0,cf_1,NaN,NaN,NaN,NaN,4.0,NaN,21.515459,NaN,0.056522,0.018602,True
2,0,cf_2,4.0,NaN,NaN,NaN,NaN,NaN,24.451119,NaN,0.056522,0.035129,False
3,0,cf_3,NaN,NaN,NaN,NaN,NaN,NaN,22.924936,NaN,0.056522,0.078589,False


In [5]:
int_columns = [
    "etfruit",
    "eatveg",
    "cgtsmok",
    "alcfreq",
    "slprl",
    "paccnois",
    "dosprt",
]

float_columns = [
    "bmi",
    "risk_before",
    "predicted_risk_after",
]

In [6]:
batch_cfcheck_df[int_columns] = batch_cfcheck_df[int_columns].astype("Int64")

In [7]:
batch_cfcheck_df[float_columns] = batch_cfcheck_df[float_columns].round(2)

batch_cfcheck_df.head(4)

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,risk_before,predicted_risk_after,valid
0,0,xin,2,3,6,3,1,0,20.44,2,0.06,NaN,NaN
1,0,cf_1,<NA>,<NA>,<NA>,<NA>,4,<NA>,21.52,<NA>,0.06,0.02,True
2,0,cf_2,4,<NA>,<NA>,<NA>,<NA>,<NA>,24.45,<NA>,0.06,0.04,False
3,0,cf_3,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,22.92,<NA>,0.06,0.08,False


In [8]:
batch_cfcheck_df[int_columns] = (
    batch_cfcheck_df[int_columns]
    .astype("string")
    .fillna("")
)

In [9]:
batch_cfcheck_df = batch_cfcheck_df.replace({np.nan : ""})
batch_cfcheck_df.head(4)

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,risk_before,predicted_risk_after,valid
0,0,xin,2,3,6,3,1,0,20.44,2,0.06,,
1,0,cf_1,,,,,4,,21.52,,0.06,0.02,True
2,0,cf_2,4,,,,,,24.45,,0.06,0.04,False
3,0,cf_3,,,,,,,22.92,,0.06,0.08,False


In [10]:
mask = batch_cfcheck_df["cf_id"] == "xin"
batch_cfcheck_df.loc[mask, "risk_before"] = ""
batch_cfcheck_df.head(4)

/tmp/ipykernel_56105/1160248817.py:2: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  batch_cfcheck_df.loc[mask, "risk_before"] = ""


,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,risk_before,predicted_risk_after,valid
0,0,xin,2,3,6,3,1,0,20.44,2,,,
1,0,cf_1,,,,,4,,21.52,,0.06,0.02,True
2,0,cf_2,4,,,,,,24.45,,0.06,0.04,False
3,0,cf_3,,,,,,,22.92,,0.06,0.08,False


In [11]:
feature_cols = ["etfruit", "eatveg", "cgtsmok", "alcfreq", "slprl", "paccnois", "bmi", "dosprt"]

# 1. Beräkna Nchanged endast för CF-rader
mask_cf = batch_cfcheck_df["cf_id"] != "xin"

batch_cfcheck_df.loc[mask_cf, "Nchanged"] = (
    batch_cfcheck_df.loc[mask_cf, feature_cols] != ""
).sum(axis=1)

# 2. Konvertera kolumnen till ren string
batch_cfcheck_df["Nchanged"] = batch_cfcheck_df["Nchanged"].astype("string")

# 3. Baseline ska vara tom
batch_cfcheck_df.loc[batch_cfcheck_df["cf_id"] == "xin", "Nchanged"] = ""

In [12]:
batch_cfcheck_df.head(4)

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,risk_before,predicted_risk_after,valid,Nchanged
0,0,xin,2,3,6,3,1,0,20.44,2,,,,
1,0,cf_1,,,,,4,,21.52,,0.06,0.02,True,2
2,0,cf_2,4,,,,,,24.45,,0.06,0.04,False,2
3,0,cf_3,,,,,,,22.92,,0.06,0.08,False,1


In [13]:
batch_cfcheck_df["Gower"] = ""
batch_cfcheck_df["Expected"] = ""


In [14]:
order = ["query_index", "cf_id"] + feature_cols + ["Gower", "Nchanged", "valid", "risk_before", "predicted_risk_after", "Expected"]

batch_cfcheck_df = batch_cfcheck_df[order]
batch_cfcheck_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,Gower,Nchanged,valid,risk_before,predicted_risk_after,Expected
0,0,xin,2,3,6,3,1,0,20.44,2,,,,,,
1,0,cf_1,,,,,4,,21.52,,,2,True,0.06,0.02,
2,0,cf_2,4,,,,,,24.45,,,2,False,0.06,0.04,
3,0,cf_3,,,,,,,22.92,,,1,False,0.06,0.08,
4,1,xin,3,4,1,2,3,0,22.38,0,,,,,,
5,1,cf_1,,,6,,,,,2,,2,False,0.25,0.2,
6,1,cf_2,,,,,,,21.37,,,1,True,0.25,0.02,
7,1,cf_3,,,6,,,,21.74,,,2,True,0.25,0.05,
8,2,xin,5,3,1,1,4,0,22.69,7,,,,,,
9,2,cf_1,,,,3,,,19.4,,,2,True,0.22,0.03,


# picking correct CF

### expected


In [15]:
# Check directional constraints directly
# These features should only improve (increase) or stay the same
SHOULD_INCREASE = ["cgtsmok", "alcfreq", "dosprt"]

# These features should only improve (decrease) or stay the same
SHOULD_DECREASE = ["bmi", "etfruit", "eatveg", "slprl", "paccnois"]


print("Directional constraints:")
print(f"  Should increase (↑): {SHOULD_INCREASE}")
print(f"  Should decrease (↓): {SHOULD_DECREASE}")


Directional constraints:
  Should increase (↑): ['cgtsmok', 'alcfreq', 'dosprt']
  Should decrease (↓): ['bmi', 'etfruit', 'eatveg', 'slprl', 'paccnois']


In [16]:
# Check if CFs violate directional constraints
# First, ensure numeric columns are actually numeric for comparison
numeric_cols = ["etfruit", "eatveg", "cgtsmok", "alcfreq", "slprl", "paccnois", "bmi", "dosprt"]

# Create a working copy with numeric values
df_check = batch_cfcheck_df.copy()
for col in numeric_cols:
    df_check[col] = pd.to_numeric(df_check[col], errors="coerce")

# Extract baseline values per query_index
baseline_values = df_check[df_check["cf_id"] == "xin"].set_index("query_index")[numeric_cols]

# Check violations for each CF
def check_violations(row):
    if row["cf_id"] == "xin":
        return ""

    baseline = baseline_values.loc[row["query_index"]]
    violations = []

    # Check features that should increase
    for feat in SHOULD_INCREASE:
        if pd.notna(row[feat]) and pd.notna(baseline[feat]):
            if row[feat] < baseline[feat]:
                violations.append(f"{feat}↓")

    # Check features that should decrease
    for feat in SHOULD_DECREASE:
        if pd.notna(row[feat]) and pd.notna(baseline[feat]):
            if row[feat] > baseline[feat]:
                violations.append(f"{feat}↑")

    return "NO: " + ", ".join(violations) if violations else ""

batch_cfcheck_df["Expected"] = df_check.apply(check_violations, axis=1)
batch_cfcheck_df.head(10)


,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,Gower,Nchanged,valid,risk_before,predicted_risk_after,Expected
0,0,xin,2,3,6,3,1,0,20.44,2,,,,,,
1,0,cf_1,,,,,4,,21.52,,,2,True,0.06,0.02,"NO: bmi↑, slprl↑"
2,0,cf_2,4,,,,,,24.45,,,2,False,0.06,0.04,"NO: bmi↑, etfruit↑"
3,0,cf_3,,,,,,,22.92,,,1,False,0.06,0.08,NO: bmi↑
4,1,xin,3,4,1,2,3,0,22.38,0,,,,,,
5,1,cf_1,,,6,,,,,2,,2,False,0.25,0.2,
6,1,cf_2,,,,,,,21.37,,,1,True,0.25,0.02,
7,1,cf_3,,,6,,,,21.74,,,2,True,0.25,0.05,
8,2,xin,5,3,1,1,4,0,22.69,7,,,,,,
9,2,cf_1,,,,3,,,19.4,,,2,True,0.22,0.03,


### is valid

In [17]:
# --- Select exactly one CF per query_index (random valid without violations) ---

# Split baseline and CF rows
df_xin = batch_cfcheck_df[batch_cfcheck_df["cf_id"] == "xin"]
df_cf  = batch_cfcheck_df[batch_cfcheck_df["cf_id"] != "xin"]

def select_random_cf(group):
    """Select one random CF that is valid and has no violations (Expected is empty)."""
    # First try: valid AND no violations
    good_cfs = group[(group["valid"] == True) & (group["Expected"] == "")]
    if len(good_cfs) > 0:
        return good_cfs.sample(n=1, random_state=42)

    # Fallback: any valid CF
    valid_cfs = group[group["valid"] == True]
    if len(valid_cfs) > 0:
        return valid_cfs.sample(n=1, random_state=42)

    # Last resort: first CF
    return group.iloc[[0]]

# Select one random CF per query_index
df_cf_selected = df_cf.groupby("query_index", group_keys=False).apply(select_random_cf).reset_index(drop=True)

# Combine baseline + selected CF
batch_cfcheck_df = pd.concat([df_xin, df_cf_selected], ignore_index=True)

# Ensure xin appears before CF for each query_index
batch_cfcheck_df["is_xin"] = (batch_cfcheck_df["cf_id"] == "xin").astype(int)
batch_cfcheck_df = (
    batch_cfcheck_df
    .sort_values(["query_index", "is_xin"], ascending=[True, False])
    .drop(columns="is_xin")
)
batch_cfcheck_df

/tmp/ipykernel_56105/2325069138.py:23: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_cf_selected = df_cf.groupby("query_index", group_keys=False).apply(select_random_cf).reset_index(drop=True)


,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,Gower,Nchanged,valid,risk_before,predicted_risk_after,Expected
0,0,xin,2,3,6,3,1,0,20.44,2,,,,,,
9,0,cf_1,,,,,4,,21.52,,,2,True,0.06,0.02,"NO: bmi↑, slprl↑"
1,1,xin,3,4,1,2,3,0,22.38,0,,,,,,
10,1,cf_3,,,6,,,,21.74,,,2,True,0.25,0.05,
2,2,xin,5,3,1,1,4,0,22.69,7,,,,,,
11,2,cf_1,,,,3,,,19.4,,,2,True,0.22,0.03,
3,3,xin,6,4,1,1,1,0,26.01,0,,,,,,
12,3,cf_1,,,,,,,23.7,7,,2,True,0.12,0.01,
4,4,xin,2,3,5,4,1,0,27.68,4,,,,,,
13,4,cf_2,4,,,,,,24.45,,,2,True,0.05,0.01,NO: etfruit↑


In [18]:
batch_cfcheck_df["valid"] = batch_cfcheck_df["valid"].replace(
    {
        False : "No",
        True : "Yes",
    }
)